## xx - Model training

### 🎯 Model objective

As stated before, the objective of the project is to have a program that takes as input two reactives in SMILES and gives back a list of the best conditions with their predicted yield for the **Suzuki coupling** reaction. The conditions include the **base, ligand, and solvent** used. For this, the program needs a model that takes as input a **vector of descriptors** containing significant information about the two reactives and gives back the wanted results.


### 🏛️ Notebook Structure

To accomplish this objective, multiple different models and strategies will be tested. Different models will be trained and compared in order to select the best one.

First, preliminary tasks will be performed. The necessary libraries and functions will be imported, and functions to evaluate and compare the models will be defined.

Then for each strategy, the section will contain :

- An explanation and the motivations for choosing the strategy.
- The creation of the required models, their training, and eventual adjustments.
- The testing of the strategy, where its accuracy will be tested.

The final strategy and model will be chosen based on **a comparative accuracy assessment**.

## Preliminary tasks

Before training and testing the models in their different strategies, a few preliminary tasks have to be done.

### 📚 Library and functions imports

A few libraries and functions will be necessary throughout the notebook. 

In [3]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import necessary functions
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, explained_variance_score
from sklearn.model_selection import GridSearchCV


### 💯 Testing function

To evaluate the models, a function that calculates multiple evaluation metrics is defined.

In [4]:
def test_model(model, x_test, y_test):
    """
    Test the model on the test dataset and return multiple evaluation metrics.
    Parameters:
    model: The trained model to be evaluated.
    x_test: The test features.
    y_test: The true labels for the test dataset.
    Returns:
    A dictionary containing the evaluation metrics: RMSE, MAE, R2 Score, and Explained Variance Score.
    """
    y_pred = model.predict(x_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    explained_variance = explained_variance_score(y_test, y_pred)

    return {
        "Model": model.__class__.__name__,
        "RMSE": rmse,
        "MAE": mae,
        "R2 Score": r2,
        "Explained Variance Score": explained_variance
    }

Here, it was decided to use the Root Mean Squared Error, the Mean Absolute Error, the R-Squared Score and the Explained Variance Score. This set of metrics involves a good equilibrium between interpretability and robustness.

## Strategy 1 : Every combination's yield prediction

This strategy is straight-forward. A model takes as input a vector of descriptors containing information about the two reactives and the combination of reaction conditions. It would then return the expected yield for this particular combination. The model would then be used to predict the yield of every combination of ligand, base, and solvent with the input reactives. The results could then be sorted and the conditions selected. 

Here is a summary of the number of ligands, bases, and solvents:

| Element | Number |
|---|---|
| **Ligands** | 11 ligands + "no ligand" |
| **Bases** | 7 bases + "no base" | 
| **Solvents** | 6 solvents |

This amounts to 576 combinations when accounting for the possibility of no ligand or base. This method may seem slow and computationnaly expensive, but testing such a low amount of combinations is reasonable, especially if it allows the use of simpler strategies.


### Models to test

The following models will be tested to apply this strategy:

- Random Forest Regressor
- XGboost Regressor

All the models are regressor since their return a single value when given a vector as input.

### Model definition and training

**Random Forest**

This model is composed of **multiple trees** that are each trained on a different subset of the training dataset. When given an input, each tree processes the input vector and returns a single value. The output predicted value is **the average of all the proposed results**.

**XGBoost**

**Extreme gradient boosting** model is constructed by buildent decision trees one after the other. Each new tree attempts to correct the error of the previous one. It uses **gradient based optimization** to improve its performance. The final tree is used to predict the value of interest.

In [7]:
# Create models with grid search for hyperparameter tuning
# Temporarily defining x_train, y_train, x_test, y_test for avoidance of errors, replace with actual training and test data
x_train = np.random.rand(100, 10)  # Replace with actual training features
y_train = np.random.rand(100)      # Replace with actual training labels
x_test = np.random.rand(20, 10)   # Replace with actual test features
y_test = np.random.rand(20)       # Replace with actual test labels

# Grid search 
param_grid = {
    'n_estimators': [10, 50, 100],
    'max_depth': [None, 10, 20],
}

# Random Forest Regressor
grid_search_ranf = GridSearchCV(estimator=RandomForestRegressor(random_state=0), param_grid=param_grid, cv=5)
grid_search_ranf.fit(x_train, y_train)
print("Best parameters for Random Forest:", grid_search_ranf.best_params_)

# XGBoost Regressor
grid_search_xgb = GridSearchCV(estimator=XGBRegressor(random_state=0), param_grid=param_grid, cv=5)
grid_search_xgb.fit(x_train, y_train)
print("Best parameters for XGBoost:", grid_search_xgb.best_params_)

Best parameters for Random Forest: {'max_depth': 10, 'n_estimators': 100}
Best parameters for XGBoost: {'max_depth': None, 'n_estimators': 10}


In [ ]:
# Test the models 
print(test_model(grid_search_ranf.best_estimator_, x_test, y_test))
print(test_model(grid_search_xgb.best_estimator_, x_test, y_test))

# Predict on the first 5 test vectors and display results
num_samples = 5
y_pred_ranf = grid_search_ranf.best_estimator_.predict(x_test[:num_samples])
y_pred_xgb = grid_search_xgb.best_estimator_.predict(x_test[:num_samples])

print("\nPredictions for first 5 test vectors:")
for i in range(num_samples):
    print(f"Tested vector {i+1}, Random Forest predicted yield: {y_pred_ranf[i]:.4f}, actual yield: {y_test[i]:.4f}")
    print(f"Tested vector {i+1}, XGBoost predicted yield: {y_pred_xgb[i]:.4f}, actual yield: {y_test[i]:.4f}")
    print("")

{'Model': 'RandomForestRegressor', 'RMSE': np.float64(0.30391979931215024), 'MAE': 0.265758667922723, 'R2 Score': -0.02464440927844591, 'Explained Variance Score': -0.004753483092195143}
{'Model': 'XGBRegressor', 'RMSE': np.float64(0.31783561856303005), 'MAE': 0.2611090943170147, 'R2 Score': -0.12062502789649576, 'Explained Variance Score': -0.11960189816465916}

Predictions for first 5 test vectors:
Tested vector 1, Random Forest predicted yield: 0.4361, actual yield: 0.9602
Tested vector 1, XGBoost predicted yield: 0.5324, actual yield: 0.9602

Tested vector 2, Random Forest predicted yield: 0.4286, actual yield: 0.7241
Tested vector 2, XGBoost predicted yield: 0.5946, actual yield: 0.7241

Tested vector 3, Random Forest predicted yield: 0.5810, actual yield: 0.4261
Tested vector 3, XGBoost predicted yield: 0.8070, actual yield: 0.4261

Tested vector 4, Random Forest predicted yield: 0.6178, actual yield: 0.9182
Tested vector 4, XGBoost predicted yield: 0.7662, actual yield: 0.9182

